# Indexing TREC Robust 2004 by OpenSearch for Sparse Encoder Model

- [disks45/nocr/trec-robust-2004](https://ir-datasets.com/disks45.html#disks45/nocr/trec-robust-2004)

### Install python modules

In [3]:
import sys
!{sys.executable} -m pip install ir_datasets pandas opensearch-py nltk sentence_transformers torch torchvision

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 12.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 20.6 MB/s eta 0:00:00
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_aarch64.manylinux2014_aarch64.whl.metadata (7.3 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-manylinux2014_aarch64.manylinux_2_17_aarch64.manylinux_2_28_aarch64.whl.metadata (2.7 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached shelli

### Load helper modules

In [4]:
import pprint
from tqdm import tqdm

### Create an OpenSearch Client

Your opensearch password should be available in `~/.env`

```bash
    OPENSEARCH_INITIAL_ADMIN_PASSWORD="strong password"
```

In [5]:
import os
from dotenv import load_dotenv
from opensearchpy import OpenSearch

load_dotenv()
host = 'localhost'
port = 9200
password = os.getenv("OPENSEARCH_INITIAL_ADMIN_PASSWORD")

client = OpenSearch(
    hosts=[{"host": host, "port": port}],
    http_auth=("admin", password),
    http_compress=True,
    use_ssl=True,
    verify_certs=False,
    ssl_assert_hostname=False,
    ssl_show_warn=False
)
pprint.pprint(client.info())

{'cluster_name': 'docker-cluster',
 'cluster_uuid': 'qIJ28Ej2TCC6_LI88zZLlw',
 'name': '4db878c40bab',
 'tagline': 'The OpenSearch Project: https://opensearch.org/',
 'version': {'build_date': '2026-02-07T07:54:31.169913465Z',
             'build_hash': 'bbc94f0bdc3a759011e6529ecfe52840856f91a3',
             'build_snapshot': False,
             'build_type': 'tar',
             'distribution': 'opensearch',
             'lucene_version': '10.3.2',
             'minimum_index_compatibility_version': '2.0.0',
             'minimum_wire_compatibility_version': '2.19.0',
             'number': '3.5.0'}}


### Index a Corpus for SPLADE Model

- Note: You should have a GPU for indexing.
- We're using an "inference-free" model which does not expand queries.
    - See [https://sbert.net/docs/sparse_encoder/pretrained_models.html](https://sbert.net/docs/sparse_encoder/pretrained_models.html)

In [6]:
import ir_datasets
dataset_name = "disks45/nocr/trec-robust-2004"
dataset = ir_datasets.load(dataset_name)
docstore = dataset.docs_store()
docstore.build()

Index structure

In [35]:
index_name = "trec_robust_2004_splade"
if client.indices.exists(index=index_name):
    client.indices.delete(index=index_name)

In [36]:
index_body ={
  "settings": {
    "index": {
      "number_of_shards": 1,
      "number_of_replicas": 0
    }
  },
  "mappings": {
    "properties": {
      "docid": { "type": "keyword" },
      "title": { "type": "text" },
      "text": { "type": "text" },
      "sparse_embedding": {
        "type": "rank_features"
      }
    }
  }
}
response = client.indices.create(index=index_name, body=index_body)
pprint.pprint(response)

{'acknowledged': True,
 'index': 'trec_robust_2004_splade',
 'shards_acknowledged': True}


Encoding Model

In [37]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [38]:
from sentence_transformers.sparse_encoder import SparseEncoder
# encoder_model = "naver/splade-cocondenser-ensembledistil"
encoder_model = "opensearch-project/opensearch-neural-sparse-encoding-doc-v3-distill"
model = SparseEncoder(encoder_model, trust_remote_code=True).to(device)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 9292.36it/s]


Indexing

- Note: No text chunking is performed. The first max tokens are indexed.

In [39]:
from bs4 import BeautifulSoup
def parse_marked_up_doc(marked_up_doc):
    # Parse the content using BeautifulSoup
    soup = BeautifulSoup(marked_up_doc, 'html.parser')
    
    # Extract the title from the <HEADLINE> tag
    headline_tag = soup.find('headline')
    title = headline_tag.get_text(strip=True) if headline_tag else "No Title"
    
    # Extract the text from the <TEXT> tag and join paragraphs
    text = ' '.join(p.get_text(strip=True) for p in soup.find_all('p'))
    
    return title, text

In [40]:
from itertools import islice
def batched(iterable, batch_size):
    iterator = iter(iterable)

    while True:
        batch = list(islice(iterator, batch_size))
        if not batch:
            break
        yield batch

In [41]:
def prepare_documents(dataset, docstore, model, batch_size=32):

    for docs_batch in batched(dataset.docs_iter(), batch_size):

        doc_infos = []
        texts_for_encoding = []

        # Prepare batch
        for doc in docs_batch:
            title, text = parse_marked_up_doc(
                docstore.get(doc.doc_id).marked_up_doc
            )
            text = text.replace("\n", " ")

            doc_infos.append({
                "doc_id": doc.doc_id,
                "title": title,
                "text": text,
            })

            texts_for_encoding.append(f"{title}\n{text}")

        # Encode entire batch at once
        doc_tensors = model.encode_document(texts_for_encoding)
        doc_embeddings = model.decode(doc_tensors)

        # Yield documents
        for info, embedding in zip(doc_infos, doc_embeddings):
            yield {
                "_id": info["doc_id"],
                "_source": {
                    "docid": info["doc_id"],
                    "title": info["title"],
                    "text": info["text"],
                    "sparse_embedding": {
                        str(k): float(v)
                        for k, v in dict(embedding).items()
                    }
                }
            }

In [42]:
from opensearchpy.helpers import bulk
total_docs = sum(1 for _ in dataset.docs_iter())
success, failed = bulk(
    client, 
    tqdm(
        prepare_documents(dataset, docstore, model, batch_size=64),
        total=total_docs,
        desc="Indexing"
    ),
    index=index_name
)

Indexing: 100%|██████████| 528155/528155 [45:03<00:00, 195.34it/s] 


#### Search Test

In [43]:
def search(query: str, size: int = 10) -> dict:
    query_tensor = model.encode_query([query])
    query_embedding = model.decode(query_tensor)
    query_body = {
        "size": size,
        "query": {
            "neural_sparse": {
                "sparse_embedding": {
                    "query_tokens": dict(query_embedding[0])
                }
            }
        }
    }
    return client.search(index=index_name, body=query_body)

In [ ]:
q = "killer bee attack human"
resp = search(q, size=5)

print(f"\nTop {len(resp['hits']['hits'])} hits for query: {q}\n")
for hit in resp["hits"]["hits"]:
    src = hit["_source"]
    print(f"[{src['docid']}] {src['title'][:50]}... (score={hit['_score']:.2f})")